In [1]:
!conda info --envs

# conda environments:
#
base                     /home/sakdag/miniconda3
fake-news-classification  *  /home/sakdag/miniconda3/envs/fake-news-classification
snorkel-env              /home/sakdag/miniconda3/envs/snorkel-env



In [2]:
!pwd

/home/sakdag/weakly-supervised-fake-news-classification/src/weak-labeling


In [3]:
import sys
import os
import datetime

import tqdm
import warnings
import transformers
import pickle

import pandas as pd
import tensorflow as tf
import numpy as np

from keras.preprocessing import sequence
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from statistics import mean
from types import SimpleNamespace

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
stop_words = set(stopwords.words('english'))

from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.apply.dask import DaskLFApplier, PandasParallelLFApplier
import dask.dataframe as dd
import dask.array as da
import dask.bag as db
from better_profanity import profanity

/home/sakdag/miniconda3/envs/fake-news-classification/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sakdag/miniconda3/envs/fake-news-classification/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.1
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/home/sakdag/miniconda3/envs/fake-news-classification/lib/python3.10/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[nltk_data] Downloading package stopwords to /home/sakdag/nltk_data...
[nltk_data]   Package stopwords is already up-

In [4]:
sys.path.append("/home/sakdag/weakly-supervised-fake-news-classification")

from src.utils.path_utils import get_project_root
from src.read_data.consts import RawDatasets
from src.read_data.read_raw_data import read_raw_data
from src.utils.preprocessing_utils import cleanup_df

covid_df: pd.DataFrame = read_raw_data(RawDatasets.COVID_DATASET_SAMPLED)
elections_df: pd.DataFrame = read_raw_data(RawDatasets.ELECTIONS_DATASET_SAMPLED)

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/raw/nela-covid-2020-sampled.csv
Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/raw/nela-elections-2022-sampled.csv


In [5]:
elections_df_cleaned = cleanup_df(elections_df)
elections_df_cleaned.head()

,title,content,label
0,The Latest U.S. Tool to Fight Election Meddlin...,An American official said the text messages we...,0
1,Trump Launches His America First Healthcare Pl...,President Donald Trump signed an executive ord...,1
2,"""We're Ready To Fight"": 1000s Expected To Atte...",As various pro-gun rights groups prepare to ga...,1
3,San Diego Bishop Says Earth Is Headed Toward ‘...,"Written by Thomas D. Williams , Ph.D. on .\nPo...",1
4,Unite Threatens to 'Review' Funding to Labour ...,The leader of the Labour 's largest trade unio...,1


In [6]:
covid_df_cleaned = cleanup_df(covid_df)
covid_df_cleaned.head()

,title,content,label
0,Trump Says NYC Falsely Inflating Virus Death F...,Speaking at the Wednesday daily White House vi...,1
1,Did coronavirus lay bare inequalities? Not to ...,"But while some people tweet , write and speak ...",0
2,France crisis: Fury erupts as hero doctors and...,We will use your email address only for sendin...,1
3,The U.S. Air Force Is Short Thousands of Pilot...,The U.S. Air Force still is short thousands of...,0
4,Irish Freedom Party chair asked to resign from...,A University College Dublin ( UCD ) professor ...,0


In [7]:
elections_train, elections_test = train_test_split(elections_df_cleaned, train_size=0.8)
print(f"{len(elections_train)} - {len(elections_test)}")

26802 - 6701


In [8]:
small_elections_train = elections_train.sample(n=100).reset_index(drop = True)
# small_elections_ddf = dd.from_pandas(small_elections_train, npartitions=100)

In [9]:
from src.labeling_functions.classifier.classifier_lfs import lf_xgboost_classifier
from src.labeling_functions.stylistic.count_based import lf_word_per_sentence
from src.labeling_functions.stylistic.pos_tagging import lf_adverb_frequency

lfs = [lf_word_per_sentence, lf_adverb_frequency, lf_xgboost_classifier]

In [ ]:
applier = PandasLFApplier(lfs)
L_train = applier.apply(elections_train)

 80%|█████████████████████████████████████████████████▋            | 21487/26802 [28:19<47:02:26, 31.86s/it]

In [ ]:
LFAnalysis(L=L_train, lfs=lfs).lf_summary(Y=elections_train.label.values)

In [13]:
import multiprocessing

cores = multiprocessing.cpu_count() # Count the number of cores in a computer
cores

12

In [10]:
from dask.distributed import Client
client = Client(n_workers=6, threads_per_worker=2, memory_limit='1GB')

2023-01-13 22:47:04,933 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-q5e6l_xr', purging
2023-01-13 22:47:04,935 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-neh5crab', purging
2023-01-13 22:47:04,935 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-tvb6c036', purging
2023-01-13 22:47:04,936 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-_k6nwj0e', purging
2023-01-13 22:47:04,936 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-alw4yo0u', purging
2023-01-13 22:47:04,937 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-p5e7dsso', purging
2023-01-13 22:47:04,937 - distributed.diskutils - INFO - Found stale lock file and directory '/tmp/dask-worker-space/worker-pgqd9vch', purging

In [11]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 6
Total threads: 12,Total memory: 5.59 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:40627,Workers: 6
Dashboard: http://127.0.0.1:8787/status,Total threads: 12
Started: Just now,Total memory: 5.59 GiB
Comm: tcp://127.0.0.1:39023,Total threads: 2
Dashboard: http://127.0.0.1:41371/status,Memory: 0.93 GiB
Nanny: tcp://127.0.0.1:44085,


In [12]:
from dask.diagnostics import ProgressBar
pbar = ProgressBar()                
pbar.register() # global registration

In [ ]:
applier = PandasParallelLFApplier(lfs)
with pbar:
  L_train = applier.apply(elections_train, n_parallel=12, scheduler=client)

In [ ]:
client.close()

In [ ]:
L_train

In [ ]:
LFAnalysis(L=L_train, lfs=lfs).lf_summary(Y=elections_train.label.values)

In [ ]:
from snorkel.labeling.model import MajorityLabelVoter, LabelModel

In [ ]:
label_model = LabelModel(cardinality=2, verbose=True)
label_model.fit(L_train=L_train, n_epochs=500, log_freq=100, seed=1234)

In [ ]:
predictions = label_model.predict(L_train)
predictions

In [ ]:
print(list(small_elections_train["label"]))

In [ ]:
predictions_proba = label_model.predict_proba(L_train)
predictions_proba

In [ ]:
matches = np.count_nonzero(small_elections_train["label"] == predictions)
mismatches = predictions.size - matches
mismatches

In [ ]:
features_2 = func_tokenizer(roberta_tokenizer, small_elections_train["content"])
padded_features_2 = utils.pad_sequences(features_2, maxlen = 500)
xgb_pred_2 = xgb.predict(padded_features_2)

In [ ]:
xgb_pred_2

In [ ]:
matches = np.count_nonzero(small_elections_train["label"] == xgb_pred_2)
mismatches = predictions.size - matches
mismatches

In [ ]:
small_elections_train.head()

In [ ]:

preds = []
for index, row in small_elections_train.iterrows():
  # print(row[1])
  preds.append(lf_adverb_frequency(x=SimpleNamespace(content=row[1])))
preds

In [ ]:
matches = np.count_nonzero(small_elections_train["label"] == preds)
mismatches = predictions.size - matches
mismatches

In [ ]:

preds = []
for index, row in small_elections_train.iterrows():
  # print(row[1])
  preds.append(lf_swear_words(x=SimpleNamespace(content=row[1])))
preds

In [ ]:
matches = np.count_nonzero(small_elections_train["label"] == preds)
mismatches = predictions.size - matches
mismatches

# Unused pieces

In [ ]:
from catboost import CatBoostClassifier
cb = CatBoostClassifier(eval_metric = 'Accuracy', iterations = 2000, learning_rate = 0.2)

In [ ]:
cb.fit(roberta_trg, y_train, verbose = 0)
cb_pred = cb.predict(roberta_test)
cb_score = accuracy_score(y_test, cb_pred)
cb_roc = roc_auc_score(y_test, cb_pred)
print("The accuracy of CatBoost is: %0.2f" %cb_score)
print("The roc_auc score of CatBoost is: %0.2f" %cb_roc)

In [ ]:
from sklearn.metrics import classification_report
cr2 = classification_report(y_test, cb_pred)
print(cr2)